# 🐍 Day 09 — Error Handling
**Phase 3: Intermediate  ·  Estimated Time: 2 hours**

---

## 🎯 Learning Objectives
By the end of this session you will be able to:
- Use `try / except / else / finally` to handle errors gracefully
- Catch specific exception types rather than using bare `except`
- Raise exceptions manually with `raise`
- Create custom exception classes by subclassing `Exception`
- Use the `logging` module as a professional alternative to `print()`

**Prerequisites:** Day 08 (File I/O), Day 05 (Functions), Day 06 (Data Structures)


## 🧠 Concepts

### Concept 1 — What Are Exceptions?

In Python, an error that occurs at runtime is called an **exception**.  
When Python raises an exception, it stops execution and shows a **traceback** — the error type, message, and line number.

Every exception is an *object* that belongs to a class hierarchy:

```
BaseException
  ├── SystemExit
  ├── KeyboardInterrupt
  └── Exception          ← the one we usually catch
        ├── ValueError   (right type, wrong value)
        ├── TypeError    (wrong type)
        ├── KeyError     (dict key missing)
        ├── IndexError   (list index out of range)
        ├── FileNotFoundError
        └── ZeroDivisionError
```

Understanding the hierarchy matters: catching a parent class catches all its children too.


In [ ]:
# Exceptions in action — each line raises a different error
# (run one at a time by uncommenting)

# ValueError: right type, but invalid value
# int("hello")

# TypeError: wrong type for the operation
# result = "age" + 25

# KeyError: key doesn't exist in the dict
# profile = {"name": "Priya"}
# print(profile["age"])

# IndexError: list index out of range
# scores = [88, 92, 75]
# print(scores[10])

# ZeroDivisionError
# result = 100 / 0

# FileNotFoundError
# open("missing.txt")

print("Each of the lines above raises a different exception type.")
print("Uncomment one at a time to see the traceback.")


### Concept 2 — try / except Basics

Wrap risky code in a `try` block. List `except` clauses for each exception type you want to handle.

**Syntax:**
```python
try:
    # code that might raise an exception
except ExceptionType as e:
    # runs only if that exception occurs
    # 'e' is the exception object
```

**Rules:**
- Always catch the most specific exception first
- Never use bare `except:` — it silently swallows `KeyboardInterrupt` and `SystemExit`
- Use `except Exception as e` as a last-resort catch-all


In [ ]:
# ── Basic try/except ─────────────────────────────────────

def safe_divide(a, b):
    try:
        result = a / b
        print(f"{a} / {b} = {result}")
        return result
    except ZeroDivisionError:
        print(f"Error: cannot divide {a} by zero")
        return None
    except TypeError as e:
        print(f"Error: wrong types provided — {e}")
        return None

safe_divide(10, 2)    # OK
safe_divide(10, 0)    # ZeroDivisionError
safe_divide(10, "x")  # TypeError


### Concept 3 — Multiple except Clauses & Tuple Syntax

You can:
1. Write separate `except` clauses for each error type  
2. Group multiple exceptions in a tuple on one line

```python
except (ValueError, TypeError) as e:
    print(f"Input error: {e}")
```


In [ ]:
# ── Catching multiple exception types ────────────────────

def parse_age(value):
    try:
        age = int(value)
        if age < 0 or age > 120:
            raise ValueError(f"Age {age} is out of realistic range")
        print(f"Valid age: {age}")
        return age
    except (ValueError, TypeError) as e:
        print(f"Invalid input: {e}")
        return None

parse_age("25")    # valid
parse_age("abc")   # ValueError
parse_age(None)    # TypeError
parse_age(-5)      # ValueError (our manual raise)
parse_age("200")   # ValueError (out of range)


### Concept 4 — else and finally

The `try` block has two optional extras:

| Clause | When it runs |
|--------|-------------|
| `else` | Only if **no** exception was raised in `try` |
| `finally` | **Always** — even if an exception occurred |

Use `else` for code that should only run on success.  
Use `finally` for cleanup — closing files, releasing locks, disconnecting from databases.


In [ ]:
# ── else and finally in action ────────────────────────────

def load_config(filename):
    file_obj = None
    try:
        file_obj = open(filename, "r", encoding="utf-8")
        content = file_obj.read()
    except FileNotFoundError:
        print(f"Config not found: {filename}")
        content = "{}"          # default empty config
    else:
        # Only runs if open() and read() succeeded
        print(f"Config loaded successfully ({len(content)} chars)")
    finally:
        # Always runs — close the file if it was opened
        if file_obj:
            file_obj.close()
            print("File handle closed")
    return content

# Try with a real file (create it first)
import tempfile, os

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    f.write('{"theme": "dark"}')
    tmp_path = f.name

result = load_config(tmp_path)
print(f"Content: {result}")
print()
result = load_config("no_such_file.txt")
print(f"Content: {result}")

os.unlink(tmp_path)   # cleanup


### Concept 5 — raise: Throwing Your Own Exceptions

Use `raise` to signal an error when Python won't automatically raise one.

```python
raise ValueError("age must be positive")     # new exception
raise                                          # re-raise the current exception
raise NewError("context") from original_err   # chained exception
```

`assert` is a shorthand for raising `AssertionError` — use it for internal invariants, **not** for validating user input (assertions can be disabled with `python -O`).


In [ ]:
# ── raise in validation functions ────────────────────────

def set_exam_score(score):
    if not isinstance(score, (int, float)):
        raise TypeError(f"Score must be a number, got {type(score).__name__}")
    if not 0 <= score <= 100:
        raise ValueError(f"Score {score} must be between 0 and 100")
    return score

# Test valid and invalid inputs
tests = [85, 110, -5, "pass", 0, 100]
for t in tests:
    try:
        result = set_exam_score(t)
        print(f"  set_exam_score({t!r:>6}) -> {result}")
    except (TypeError, ValueError) as e:
        print(f"  set_exam_score({t!r:>6}) -> ERROR: {e}")


### Concept 6 — Custom Exception Classes

Create domain-specific exception classes by subclassing `Exception` (or another exception class).

Benefits:
- Callers can catch your specific error type precisely
- You can attach extra attributes (e.g. the invalid value, error code)
- One base class per module lets callers catch all module errors in one `except`


In [ ]:
# ── Custom exception hierarchy ────────────────────────────

class ShopError(Exception):
    pass  # Base class for all online shop errors

class OutOfStockError(ShopError):
    # Raised when an item has no remaining stock
    def __init__(self, item_name, requested):
        self.item_name = item_name
        self.requested = requested
        super().__init__(f"'{item_name}' is out of stock (requested {requested})")

class InvalidCouponError(ShopError):
    # Raised when an unrecognised coupon code is used
    def __init__(self, code):
        self.code = code
        super().__init__(f"Coupon '{code}' is invalid or expired")


# ── Usage ─────────────────────────────────────────────────
INVENTORY = {"pen": 5, "notebook": 0, "eraser": 12}
COUPONS = {"SAVE10", "WELCOME"}

def place_order(item, qty, coupon=None):
    if coupon and coupon not in COUPONS:
        raise InvalidCouponError(coupon)
    if INVENTORY.get(item, 0) < qty:
        raise OutOfStockError(item, qty)
    INVENTORY[item] -= qty
    print(f"Order placed: {qty}x {item}")

orders = [
    ("pen", 2, None),
    ("notebook", 1, None),   # OutOfStockError
    ("eraser", 3, "FAKE50"), # InvalidCouponError
    ("eraser", 3, "SAVE10"), # OK
]

for item, qty, coupon in orders:
    try:
        place_order(item, qty, coupon)
    except OutOfStockError as e:
        print(f"Stock error: {e}  (item={e.item_name})")
    except InvalidCouponError as e:
        print(f"Coupon error: {e}  (code={e.code})")


### Concept 7 — The logging Module

The `logging` module is the professional replacement for `print()` debug statements.

**Why use logging?**
- Severity levels let you filter noise in production
- Timestamps and module names appear automatically
- Logs can be written to files without changing your code
- Turn off `DEBUG` messages in production with one config change

**Five levels (lowest → highest):**

| Level | When to use |
|-------|-------------|
| `DEBUG` | Detailed diagnostic info during development |
| `INFO` | Normal events — server started, file loaded |
| `WARNING` | Unexpected but recoverable — disk nearly full |
| `ERROR` | A function failed; some feature is broken |
| `CRITICAL` | Application may crash; data at risk |


In [ ]:
# ── Logging basics ────────────────────────────────────────
import logging

# One-time setup: call basicConfig() ONCE at the start of your script
logging.basicConfig(
    level=logging.DEBUG,                             # show all levels >= DEBUG
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger(__name__)   # logger named after this module

# Five log levels
logger.debug("Loaded 42 records from database")         # dev detail
logger.info("Application started on port 8000")         # normal event
logger.warning("Config file missing — using defaults")  # unexpected but ok
logger.error("Failed to connect to payment gateway")    # real problem
logger.critical("Out of disk space — shutting down")    # fatal

# Logging inside exception handlers
try:
    result = 10 / 0
except ZeroDivisionError as e:
    logger.error("Division failed: %s", e)   # use % formatting for perf
    result = None

print(f"Result: {result}")


## 🔍 Guided Examples

### Example 1 — Safe Integer Input (retry loop)


In [ ]:
# ── Retry until valid integer is entered ─────────────────
# (Using pre-assigned values to simulate interactive input)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
logger = logging.getLogger("input_demo")

def get_positive_int(prompt, simulated_inputs):
    """Keep asking until user enters a positive integer."""
    for attempt, user_input in enumerate(simulated_inputs, 1):
        try:
            value = int(user_input)
            if value <= 0:
                raise ValueError("Value must be positive")
            logger.info("Accepted value %d after %d attempt(s)", value, attempt)
            return value
        except ValueError as e:
            logger.warning("Attempt %d failed ('%s'): %s", attempt, user_input, e)
            print(f"  Try again: {e}")
    raise RuntimeError("Too many failed attempts")

# Simulate: bad input, negative, then valid
inputs = ["hello", "-5", "42"]
result = get_positive_int("Enter positive number: ", inputs)
print(f"Final value: {result}")


### Example 2 — JSON Config Loader with Fallback

In [ ]:
import json, logging, tempfile, os
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
logger = logging.getLogger("config")

DEFAULT_CONFIG = {"debug": False, "max_retries": 3, "timeout": 30}

def load_config(path):
    """Load JSON config; fall back to defaults on any error."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        logger.info("Config loaded from %s", path)
        return {**DEFAULT_CONFIG, **data}  # merge: file overrides defaults
    except FileNotFoundError:
        logger.warning("Config file not found at %s — using defaults", path)
    except json.JSONDecodeError as e:
        logger.error("Invalid JSON in %s: %s — using defaults", path, e)
    return DEFAULT_CONFIG.copy()

# Test 1: valid config file
tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"debug": True, "timeout": 60}, tmp)
tmp.close()
cfg = load_config(tmp.name)
print("Valid config:", cfg)
os.unlink(tmp.name)

# Test 2: missing file
cfg = load_config("/no/such/config.json")
print("Missing file config:", cfg)

# Test 3: broken JSON
bad = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
bad.write("{debug: True}")   # invalid JSON
bad.close()
cfg = load_config(bad.name)
print("Bad JSON config:", cfg)
os.unlink(bad.name)


### Example 3 — Custom Exception with Extra Attributes

In [ ]:
# ── Train booking system with rich custom exceptions ─────

class BookingError(Exception):
    pass

class SeatNotAvailableError(BookingError):
    def __init__(self, train_no, seat_class, requested, available):
        self.train_no  = train_no
        self.seat_class = seat_class
        self.requested  = requested
        self.available  = available
        super().__init__(
            f"Train {train_no}: only {available} '{seat_class}' seats available, "
            f"you requested {requested}"
        )

class InvalidPassengerError(BookingError):
    def __init__(self, field, value, reason):
        self.field  = field
        self.value  = value
        super().__init__(f"Invalid {field} '{value}': {reason}")


TRAINS = {
    "12301": {"Sleeper": 10, "3AC": 4, "2AC": 2},
    "22691": {"Sleeper": 0,  "3AC": 8, "2AC": 5},
}

def book_ticket(train_no, seat_class, passengers):
    if train_no not in TRAINS:
        raise BookingError(f"Train {train_no} not found")
    available = TRAINS[train_no].get(seat_class, 0)
    if available < passengers:
        raise SeatNotAvailableError(train_no, seat_class, passengers, available)
    if not (1 <= passengers <= 6):
        raise InvalidPassengerError("passengers", passengers, "must be 1–6")
    TRAINS[train_no][seat_class] -= passengers
    print(f"Booked {passengers} x {seat_class} on Train {train_no}")


bookings = [
    ("12301", "Sleeper", 3),
    ("22691", "Sleeper", 2),   # no seats
    ("12301", "1AC", 1),       # class missing
    ("99999", "Sleeper", 1),   # train not found
]

for train, cls, pax in bookings:
    try:
        book_ticket(train, cls, pax)
    except SeatNotAvailableError as e:
        print(f"  No seats: {e} (avail={e.available})")
    except BookingError as e:
        print(f"  Booking failed: {e}")


### Example 4 — Logging to File and Console

In [ ]:
import logging, tempfile, os

def setup_logger(name, log_file, console_level=logging.WARNING):
    """Return a logger writing DEBUG+ to file and WARNING+ to console."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    fmt = logging.Formatter("%(asctime)s %(name)s %(levelname)s %(message)s", "%H:%M:%S")

    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    ch = logging.StreamHandler()
    ch.setLevel(console_level)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

log_path = tempfile.mktemp(suffix=".log")
logger = setup_logger("demo", log_path)

logger.debug("Debug detail — goes to file only")
logger.info("Server started — goes to file only")
logger.warning("Memory usage at 80% — shown in console AND file")
logger.error("Database timeout — shown in console AND file")

# Read the log file and show its contents
with open(log_path) as f:
    print("\n── Contents of log file ──")
    print(f.read())

os.unlink(log_path)


## ✏️ Practice Exercises

### Exercise 1 — Safe Division Function 🟢
Write `safe_divide(a, b)` that returns the result, or `None` if `b` is zero. Catch `ZeroDivisionError` and `TypeError`. Print a helpful message for each error.


In [ ]:
# YOUR CODE HERE

def safe_divide(a, b):
    pass

# Test
print(safe_divide(10, 2))    # Expected output: 5.0
print(safe_divide(10, 0))    # Expected output: None  (with error message)
print(safe_divide(10, "x"))  # Expected output: None  (with error message)


In [ ]:
# ── Solution ──────────────────────────────────────────────
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: cannot divide by zero")
        return None
    except TypeError as e:
        print(f"Error: wrong types — {e}")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))
print(safe_divide(10, "x"))


### Exercise 2 — Dictionary Safe Get 🟢
Write `safe_get(data, key, default=None)` that returns `data[key]` or `default` if the key is missing. Use `except KeyError`.


In [ ]:
# YOUR CODE HERE
def safe_get(data, key, default=None):
    pass

student = {"name": "Arjun", "grade": "A", "score": 92}
print(safe_get(student, "name"))      # Arjun
print(safe_get(student, "age"))       # None
print(safe_get(student, "age", 18))   # 18


In [ ]:
# ── Solution ──────────────────────────────────────────────
def safe_get(data, key, default=None):
    try:
        return data[key]
    except KeyError:
        return default

student = {"name": "Arjun", "grade": "A", "score": 92}
print(safe_get(student, "name"))
print(safe_get(student, "age"))
print(safe_get(student, "age", 18))


### Exercise 3 — Input Validator with else/finally 🟡
Write `parse_percentage(value)` that:
- Converts `value` to float in `try`
- Raises `ValueError` if not between 0 and 100
- Prints "Conversion failed" in `except`
- Prints "Conversion succeeded" in `else`
- Prints "Done" in `finally` (always)


In [ ]:
# YOUR CODE HERE
def parse_percentage(value):
    pass

parse_percentage("85.5")   # succeeded, Done
parse_percentage("abc")    # failed, Done
parse_percentage("110")    # failed (out of range), Done


In [ ]:
# ── Solution ──────────────────────────────────────────────
def parse_percentage(value):
    try:
        pct = float(value)
        if not 0 <= pct <= 100:
            raise ValueError(f"{pct} is not between 0 and 100")
        result = pct
    except ValueError as e:
        print(f"Conversion failed: {e}")
        result = None
    else:
        print(f"Conversion succeeded: {result}%")
    finally:
        print("Done")
    return result

parse_percentage("85.5")
print()
parse_percentage("abc")
print()
parse_percentage("110")


### Exercise 4 — Custom Exception: TemperatureError 🟡
Create `TemperatureError(ValueError)` with attributes `value` and `scale`.  
Write `validate_temperature(value, scale)` that raises it when:
- Celsius below -273.15
- Fahrenheit below -459.67
- `scale` is not `'C'` or `'F'`


In [ ]:
# YOUR CODE HERE

class TemperatureError(ValueError):
    pass

def validate_temperature(value, scale):
    pass

tests = [(25, 'C'), (-300, 'C'), (98.6, 'F'), (-500, 'F'), (20, 'K')]
for val, sc in tests:
    try:
        validate_temperature(val, sc)
        print(f"  {val}{sc} is valid")
    except TemperatureError as e:
        print(f"  Error: {e}")


In [ ]:
# ── Solution ──────────────────────────────────────────────
class TemperatureError(ValueError):
    def __init__(self, value, scale, reason):
        self.value = value
        self.scale = scale
        super().__init__(f"{value}{scale}: {reason}")

def validate_temperature(value, scale):
    if scale not in ('C', 'F'):
        raise TemperatureError(value, scale, f"Unknown scale '{scale}', use 'C' or 'F'")
    if scale == 'C' and value < -273.15:
        raise TemperatureError(value, scale, f"Below absolute zero ({value} < -273.15)")
    if scale == 'F' and value < -459.67:
        raise TemperatureError(value, scale, f"Below absolute zero ({value} < -459.67)")

tests = [(25, 'C'), (-300, 'C'), (98.6, 'F'), (-500, 'F'), (20, 'K')]
for val, sc in tests:
    try:
        validate_temperature(val, sc)
        print(f"  {val}{sc} is valid")
    except TemperatureError as e:
        print(f"  Error: {e}")


### Exercise 5 — File Line Counter with Error Handling 🟡
Write `count_lines(filepath)` that returns the number of lines, or `-1` on error. Log each case:
- `INFO` on success  
- `ERROR` on `FileNotFoundError`  
- `ERROR` on `PermissionError`


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("counter")

# YOUR CODE HERE
def count_lines(filepath):
    pass

import tempfile, os
tmp = tempfile.NamedTemporaryFile(mode="w", delete=False)
tmp.write("line one\nline two\nline three")
tmp.close()

print(count_lines(tmp.name))                  # 3
print(count_lines("/no/such/file.txt"))        # -1
os.unlink(tmp.name)


In [ ]:
# ── Solution ──────────────────────────────────────────────
import logging, tempfile, os
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("counter")

def count_lines(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            lines = f.readlines()
        logger.info("Read %d lines from %s", len(lines), filepath)
        return len(lines)
    except FileNotFoundError:
        logger.error("File not found: %s", filepath)
        return -1
    except PermissionError:
        logger.error("Permission denied: %s", filepath)
        return -1

tmp = tempfile.NamedTemporaryFile(mode="w", delete=False)
tmp.write("line one\nline two\nline three")
tmp.close()
print(count_lines(tmp.name))
print(count_lines("/no/such/file.txt"))
os.unlink(tmp.name)


### Exercise 6 — Exception Chaining 🔴
Write `load_user_score(data, user_id)` where:
- `data` is a dict: `{user_id: "score_string"}`
- Convert the score string to int
- If `user_id` is missing, raise `LookupError` from the `KeyError`
- If the score is not a valid int, raise `ValueError` from the original error
- Include both original and new error info in the message


In [ ]:
# YOUR CODE HERE
def load_user_score(data, user_id):
    pass

scores_db = {"U001": "95", "U002": "invalid", "U003": "88"}

for uid in ["U001", "U002", "U999", "U003"]:
    try:
        score = load_user_score(scores_db, uid)
        print(f"  {uid}: score = {score}")
    except (LookupError, ValueError) as e:
        print(f"  {uid}: {type(e).__name__}: {e}")
        if e.__cause__:
            print(f"       caused by: {type(e.__cause__).__name__}: {e.__cause__}")


In [ ]:
# ── Solution ──────────────────────────────────────────────
def load_user_score(data, user_id):
    try:
        raw = data[user_id]
    except KeyError as e:
        raise LookupError(f"User '{user_id}' not found in database") from e
    try:
        return int(raw)
    except ValueError as e:
        raise ValueError(f"Score for '{user_id}' is not a valid integer: {raw!r}") from e

scores_db = {"U001": "95", "U002": "invalid", "U003": "88"}

for uid in ["U001", "U002", "U999", "U003"]:
    try:
        score = load_user_score(scores_db, uid)
        print(f"  {uid}: score = {score}")
    except (LookupError, ValueError) as e:
        print(f"  {uid}: {type(e).__name__}: {e}")
        if e.__cause__:
            print(f"       caused by: {type(e.__cause__).__name__}: {e.__cause__}")


## 🚀 Mini-Project — Student Grade Manager

Build a `GradeBook` class that:
1. Stores student names and scores (dict)
2. Raises custom exceptions for invalid data
3. Logs all operations using the `logging` module
4. Supports: `add_student`, `update_score`, `get_grade`, `class_average`, `export_summary`

**Custom exceptions to create:**
- `GradeBookError` (base)
- `StudentNotFoundError(GradeBookError)`
- `InvalidScoreError(GradeBookError)` — must be 0–100

**Grading scale:** 90–100 = A, 80–89 = B, 70–79 = C, 60–69 = D, below 60 = F


In [ ]:
import logging, json

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("gradebook")

# ── Custom exceptions ─────────────────────────────────────
class GradeBookError(Exception): pass

class StudentNotFoundError(GradeBookError):
    def __init__(self, name):
        self.name = name
        super().__init__(f"Student '{name}' not found in grade book")

class InvalidScoreError(GradeBookError):
    def __init__(self, name, score):
        self.name  = name
        self.score = score
        super().__init__(f"Score {score} for '{name}' must be between 0 and 100")


# ── GradeBook class ───────────────────────────────────────
class GradeBook:
    GRADE_SCALE = [(90, 'A'), (80, 'B'), (70, 'C'), (60, 'D'), (0, 'F')]

    def __init__(self, subject):
        self.subject = subject
        self._scores = {}
        logger.info("GradeBook created for subject: %s", subject)

    def _validate_score(self, name, score):
        if not isinstance(score, (int, float)):
            raise InvalidScoreError(name, score)
        if not 0 <= score <= 100:
            raise InvalidScoreError(name, score)

    def add_student(self, name, score):
        self._validate_score(name, score)
        self._scores[name] = score
        logger.info("Added %s with score %.1f", name, score)

    def update_score(self, name, score):
        if name not in self._scores:
            raise StudentNotFoundError(name)
        self._validate_score(name, score)
        old = self._scores[name]
        self._scores[name] = score
        logger.info("Updated %s: %.1f -> %.1f", name, old, score)

    def get_grade(self, name):
        if name not in self._scores:
            raise StudentNotFoundError(name)
        score = self._scores[name]
        for threshold, letter in self.GRADE_SCALE:
            if score >= threshold:
                return letter, score

    def class_average(self):
        if not self._scores:
            raise GradeBookError("No students in grade book")
        avg = sum(self._scores.values()) / len(self._scores)
        logger.info("Class average: %.2f", avg)
        return avg

    def export_summary(self):
        summary = []
        for name, score in sorted(self._scores.items()):
            grade, _ = self.get_grade(name)
            summary.append({"name": name, "score": score, "grade": grade})
        return summary


# ── Test the GradeBook ────────────────────────────────────
gb = GradeBook("Mathematics")

students = [
    ("Priya Sharma",  92),
    ("Arjun Patel",   78),
    ("Kavya Reddy",   85),
    ("Rohan Mehta",   61),
    ("Sneha Joshi",   55),
]

for name, score in students:
    try:
        gb.add_student(name, score)
    except InvalidScoreError as e:
        logger.error("Could not add student: %s", e)

# Test error cases
error_cases = [
    ("Unknown Student", None),   # StudentNotFoundError
    ("Priya Sharma", 110),       # InvalidScoreError (update)
]

for name, new_score in error_cases:
    try:
        if new_score is not None:
            gb.update_score(name, new_score)
        else:
            gb.get_grade(name)
    except StudentNotFoundError as e:
        print(f"Not found: {e}")
    except InvalidScoreError as e:
        print(f"Invalid score: {e}")

# Print class summary
print(f"\nClass Average: {gb.class_average():.1f}")
print("\nGrade Summary:")
for row in gb.export_summary():
    print(f"  {row['name']:<18} {row['score']:>5.1f}  Grade: {row['grade']}")


## 📋 Summary & Next Steps

### What We Covered Today

| Concept | Key Syntax | Purpose |
|---------|-----------|---------|
| Basic exception handling | `try / except ExcType as e` | Catch and respond to errors |
| Multiple exceptions | `except (TypeA, TypeB) as e` | Handle several errors together |
| Success path | `else:` | Code that runs only when no error occurs |
| Guaranteed cleanup | `finally:` | Always executes — close resources |
| Raise manually | `raise ValueError("msg")` | Signal invalid state from your own code |
| Exception chaining | `raise NewErr from original` | Preserve context when converting errors |
| Custom exceptions | `class MyErr(Exception):` | Domain-specific, attribute-rich error types |
| Logging setup | `logging.basicConfig(...)` | Replace debug `print()` statements |
| Log levels | `DEBUG INFO WARNING ERROR CRITICAL` | Filter output by severity |

### Key Takeaways
- **Always catch specific exceptions** — bare `except:` is an anti-pattern
- **Custom exceptions belong to the caller** — they let API consumers be precise
- **`finally` is for cleanup** — always close files, connections, locks here
- **Use `logging`, not `print()`** — it's free configuration control in production

### 👀 Coming Up Next — Day 10: OOP I — Classes & Objects
Tomorrow we start Object-Oriented Programming. You'll learn to:
- Define classes with `class` and the `__init__` constructor
- Understand `self` and instance variables
- Distinguish instance methods from class methods
- Spot the OOP patterns already hiding in today's code!
